In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. 載入資料與 Rank Normalization
---

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata, pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# ── 路徑設定（Colab 請改成 Drive 路徑）──────────────────────────
PATH = '/content/drive/MyDrive/Kidney_Methylation/'   # 本機；Colab 改成 '/content/drive/MyDrive/Kidney_Methylation/'

def load_methyl(cg_path, eg_path):
    df_cg = pd.read_csv(cg_path, index_col=0).T
    df_eg = pd.read_csv(eg_path, index_col=0).T
    df_cg.columns = df_cg.columns.str.lower()
    df_eg.columns = df_eg.columns.str.lower()
    common = df_cg.columns.intersection(df_eg.columns)
    X = pd.concat([df_cg[common], df_eg[common]], axis=0).fillna(0)
    y = np.concatenate([np.zeros(df_cg.shape[0]), np.ones(df_eg.shape[0])])
    return X, y

def rank_norm(Xdf):
    """Per-sample rank normalization: removes cohort-level scale offsets."""
    Xv = Xdf.values.astype(float)
    out = np.array([rankdata(row) / len(row) for row in Xv])
    return pd.DataFrame(out, index=Xdf.index, columns=Xdf.columns)

X_us, y_us = load_methyl(PATH + 'usa_CG_filtered.csv', PATH + 'usa_EG_filtered.csv')
X_jp, y_jp = load_methyl(PATH + 'jp_CG_filtered.csv',  PATH + 'jp_EG_filtered.csv')

X_us_r = rank_norm(X_us)
X_jp_r = rank_norm(X_jp)

print(f'US: {X_us.shape[0]} 人, {X_us.shape[1]} CpG  |  Normal={int((y_us==0).sum())}, CKD={int((y_us==1).sum())}')
print(f'JP: {X_jp.shape[0]} 人, {X_jp.shape[1]} CpG  |  Normal={int((y_jp==0).sum())}, CKD={int((y_jp==1).sum())}')

US: 24 人, 127856 CpG  |  Normal=12, CKD=12
JP: 33 人, 2235 CpG  |  Normal=18, CKD=15


## 2. 找兩國 DMP 一致性 CpG
---

In [ ]:
df_dmp_us = pd.read_csv(PATH + 'usa_DMP_result_TC.csv', index_col=0)
df_dmp_jp = pd.read_csv(PATH + 'jp_DMP_result_TC.csv',  index_col=0)
df_dmp_us.index = df_dmp_us.index.str.lower()
df_dmp_jp.index = df_dmp_jp.index.str.lower()

# 顯著篩選：adj.P < 0.05, |deltaBeta| > 0.1
us_sig = df_dmp_us[
    (df_dmp_us['N_to_C.adj.P.Val'] < 0.05) &
    (df_dmp_us['N_to_C.deltaBeta'].abs() > 0.1)
]
jp_sig = df_dmp_jp[
    (df_dmp_jp['N_to_C.adj.P.Val'] < 0.05) &
    (df_dmp_jp['N_to_C.deltaBeta'].abs() > 0.1)
]

common_sig = set(us_sig.index).intersection(set(jp_sig.index))
common_list = list(common_sig)
us_d = df_dmp_us.loc[common_list, 'N_to_C.deltaBeta']
jp_d = df_dmp_jp.loc[common_list, 'N_to_C.deltaBeta']

# 篩方向一致
same_dir = np.sign(us_d.values) == np.sign(jp_d.values)
concordant = np.array(common_list)[same_dir]
feats = [c for c in concordant if c in X_us_r.columns and c in X_jp_r.columns]

# 效應量相關性
r, p = pearsonr(us_d[concordant].values, jp_d[concordant].values)

print(f'US 顯著 CpG: {len(us_sig)}')
print(f'JP 顯著 CpG: {len(jp_sig)}')
print(f'兩國共同顯著: {len(common_sig)}')
print(f'方向一致（concordant）: {len(concordant)}')
print(f'兩國甲基化資料皆有: {len(feats)}')
print(f'\ndeltaBeta 相關性 (Pearson r): {r:.3f}, p={p:.2e}')
print('→ r > 0.9 代表兩國甲基化效應高度一致，訊號可信。')

US 顯著 CpG: 46964
JP 顯著 CpG: 2140
兩國共同顯著: 542
方向一致（concordant）: 28
兩國甲基化資料皆有: 17

deltaBeta 相關性 (Pearson r): 0.872, p=1.48e-09
→ r > 0.9 代表兩國甲基化效應高度一致，訊號可信。


## 3. 預測模型評估
---

In [ ]:
def nested_loo(X_df, y, feats, label):
    Xv = X_df[feats].values
    preds = []
    for i in range(len(y)):
        tr = [j for j in range(len(y)) if j != i]
        Xtr, ytr = Xv[tr], y[tr]
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
        clf = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
        clf.fit((Xtr - mu) / sd, ytr)
        preds.append(clf.predict(((Xv[[i]] - mu) / sd))[0])
    preds = np.array(preds)
    print(f'\n{"+"*50}')
    print(f'{label}')
    print(f'特徵數: {len(feats)}, 樣本數: {len(y)}')
    print(f'LOO-CV 準確率: {accuracy_score(y, preds):.4f}  ({int((preds==y).sum())}/{len(y)})')
    print(classification_report(y, preds, target_names=['Normal','CKD'], zero_division=0))
    return preds

def cross_cohort(X_tr_df, y_tr, X_te_df, y_te, feats, label):
    Xtr = X_tr_df[feats].values
    Xte = X_te_df[feats].values
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
    clf = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
    clf.fit((Xtr - mu) / sd, y_tr)
    preds = clf.predict((Xte - mu) / sd)
    print(f'\n{"+"*50}')
    print(f'{label}')
    print(f'特徵數: {len(feats)}, 訓練: {len(y_tr)}, 測試: {len(y_te)}')
    print(f'準確率: {accuracy_score(y_te, preds):.4f}  ({int((preds==y_te).sum())}/{len(y_te)})')
    print(classification_report(y_te, preds, target_names=['Normal','CKD'], zero_division=0))
    return preds

print('=' * 50)
print('模型：Logistic Regression (C=0.1)')
print('特徵：兩國 DMP 一致性 CpG（adj.P<0.05, |ΔBeta|>0.1, 方向一致）')
print('前處理：Per-sample rank normalization')
print('=' * 50)

nested_loo(X_us_r, y_us, feats, 'Within-US  Nested LOO-CV')
nested_loo(X_jp_r, y_jp, feats, 'Within-JP  Nested LOO-CV')
cross_cohort(X_us_r, y_us, X_jp_r, y_jp, feats, 'Cross-cohort: Train US → Test JP')
cross_cohort(X_jp_r, y_jp, X_us_r, y_us, feats, 'Cross-cohort: Train JP → Test US')

模型：Logistic Regression (C=0.1)
特徵：兩國 DMP 一致性 CpG（adj.P<0.05, |ΔBeta|>0.1, 方向一致）
前處理：Per-sample rank normalization

++++++++++++++++++++++++++++++++++++++++++++++++++
Within-US  Nested LOO-CV
特徵數: 17, 樣本數: 24
LOO-CV 準確率: 1.0000  (24/24)
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00        12
         CKD       1.00      1.00      1.00        12

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24


++++++++++++++++++++++++++++++++++++++++++++++++++
Within-JP  Nested LOO-CV
特徵數: 17, 樣本數: 33
LOO-CV 準確率: 0.8182  (27/33)
              precision    recall  f1-score   support

      Normal       0.77      0.94      0.85        18
         CKD       0.91      0.67      0.77        15

    accuracy                           0.82        33
   macro avg       0.84      0.81      0.81        33
weighted avg       0.83      0.82      0.81

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1.])

## 4. 一致性 CpG → 基因 → Enrichr
---

In [ ]:
gene_col = 'N_to_C.gene'

# 摘要表
summary = pd.DataFrame({
    'CpG':          feats,
    'US_deltaBeta': df_dmp_us.loc[feats, 'N_to_C.deltaBeta'].values,
    'US_adjP':      df_dmp_us.loc[feats, 'N_to_C.adj.P.Val'].values,
    'JP_deltaBeta': df_dmp_jp.loc[feats, 'N_to_C.deltaBeta'].values,
    'JP_adjP':      df_dmp_jp.loc[feats, 'N_to_C.adj.P.Val'].values,
    'Gene':         df_dmp_us.loc[feats, gene_col].values,
}).sort_values('US_adjP')

print(f'一致性 CpG 摘要（共 {len(summary)} 個）')
print(summary.to_string(index=False))

# Enrichr 基因清單
genes = (
    summary['Gene'].dropna()
    .apply(lambda x: str(x).split(';')[0].strip())
    .unique().tolist()
)
genes = [g for g in genes if g and g != 'nan']

print(f'\n對應基因數: {len(genes)}')
print('\n' + '='*40)
print('Enrichr 輸入基因清單')
print('='*40)
for g in genes:
    print(g)

# 儲存
# summary.to_csv(PATH + 'concordant_CpGs.csv', index=False)
# with open(PATH + 'enrichr_genes.txt', 'w') as f:
#     f.write('\n'.join(genes))

一致性 CpG 摘要（共 17 個）
       CpG  US_deltaBeta      US_adjP  JP_deltaBeta  JP_adjP       Gene
cg12293460      0.112191 5.443274e-10      0.219029 0.037442    SLC43A2
cg02692177      0.126364 6.878674e-07      0.224212 0.028644        NaN
cg19437852      0.140966 8.579858e-07      0.211637 0.046500       EPN1
cg01943179      0.126668 1.154969e-06      0.220036 0.029646     LPCAT1
cg21646082      0.181157 2.314452e-06      0.276321 0.029646     CCDC21
cg02823530      0.116418 3.292903e-06      0.253664 0.011541       IDH2
cg23037321     -0.105973 3.991762e-06     -0.243854 0.028188        MR1
cg04145927      0.179004 2.298487e-05      0.236407 0.049829 PARD6G-AS1
cg05193757      0.175877 7.747416e-05      0.206912 0.046500    DSCAML1
cg08250031      0.109917 1.749230e-04      0.209315 0.038278     MEGF11
cg13377224      0.123557 2.755149e-04      0.238772 0.014695     ZBTB16
cg22101188      0.113580 3.930321e-04      0.214686 0.030664      CGNL1
cg12481180      0.110293 8.517528e-04      0.